# 🤖 Notebook: Agents

In this notebook we build agents with LangChain, connect one to our MCP server, look at the ReAct pattern, and add human approval to sensitive actions.

## 📚 Sources

- [LangChain: Agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain: `create_react_agent` (classic)](https://reference.langchain.com/python/langchain-classic/agents/react/agent/create_react_agent)
- Yao et al. (2022), ["ReAct: Synergizing Reasoning and Acting in Language Models"](https://arxiv.org/pdf/2210.03629)
- [LangChain: MCP Adapters](https://docs.langchain.com/oss/python/integrations/mcp)

---

Good luck building agents! 🤗

## What is an Agent?

You've actually already built the core of an agent by hand: back in the function calling and MCP notebooks, we wrote a `while True:` loop that (1) calls the model, (2) checks whether it asked for a tool call, (3) executes that tool, (4) feeds the result back, and repeats until the model gives a final answer instead of another tool call.

That loop *is* an agent. A common way to put it:

> **Agent = Model + Harness.** The harness is everything around the loop: the model, its prompt, its tools, and anything that shapes its behavior along the way.

LangChain's [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent) builds exactly this loop for you — plus a large, optional toolbox ("middleware") for things production agents typically need: approval steps, memory, context management, retries, and more. We'll use several of these below.

One setup note: LangChain lets you select hosted models with a short string like `"openai:gpt-5.5"`. Since we're talking to our *own* Ollama server (not the default `localhost`), we'll instead construct a `ChatOllama` object ourselves and pass that in directly — `create_agent` accepts either.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads LLM_HOST from a .env file in the project root (see notebook 03 / setup.md)

LLM_HOST = os.environ["LLM_HOST"]  # the IP address you got in the lecture
LLM_URL = f"http://{LLM_HOST}:11434"
LLM_REASONING = "gemma4:26b"  # the reasoning MoE model - also supports tool calling

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model=LLM_REASONING, base_url=LLM_URL, temperature=0)

## 1. Your First LangChain Agent

`@tool` turns a Python function into a LangChain tool - the same idea as `@mcp.tool()` in the MCP notebook, just for a tool that lives locally instead of on a server. `create_agent` then builds the full loop around it.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def get_temperature(city: str) -> str:
    """Get the current temperature for a city."""
    temperatures = {"New York": "22°C", "London": "15°C", "Tokyo": "18°C"}
    return temperatures.get(city, "Unknown")


agent = create_agent(model=llm, tools=[get_temperature])

result = agent.invoke({"messages": [{"role": "user", "content": "What is the temperature in Tokyo?"}]})
for message in result["messages"]:
    print(f"{type(message).__name__}: {message.content!r}")

That's the entire loop from the function calling notebook, condensed into two lines: define a tool, call `create_agent`. `result["messages"]` holds the full transcript - the same `HumanMessage` → `AIMessage` (with a tool call) → `ToolMessage` → `AIMessage` (final answer) pattern we built by hand before.

#### What does this look like raw?

Back in the prompting notebook, we rendered Gemma 4's chat template by hand for a plain question and answer:

```md
<bos><|turn>user
What is the capital of France?<turn|>
<|turn>model
The capital of France is **Paris**.<turn|>
```

Tools and thinking extend that same template with a few more special tokens. Rendering the exact exchange above (the `get_temperature` tool definition, the model's `reasoning`, its tool call, the tool's response, and the final answer) through Gemma 4's chat template looks like this:

```md
<bos><|turn>system
<|think|>
<|tool>declaration:get_temperature{description:<|"|>Get the current temperature for a city.<|"|>,parameters:{properties:{city:{description:<|"|>The name of the city<|"|>,type:<|"|>STRING<|"|>}},required:[<|"|>city<|"|>],type:<|"|>OBJECT<|"|>}}<tool|><turn|>
<|turn>user
What is the temperature in Tokyo?<turn|>
<|turn>model
<|channel>thought
The user is asking for the temperature in Tokyo. The get_temperature tool takes a city argument and returns the temperature. This fits the request.
<channel|><|tool_call>call:get_temperature{city:<|"|>Tokyo<|"|>}<tool_call|><|tool_response>response:get_temperature{value:<|"|>18°C<|"|>}<tool_response|>The current temperature in Tokyo is 18°C.<turn|>
```

A few things worth pointing out:

- The `tools` list we pass to `create_agent` doesn't create new messages - it gets rendered once, up front, into a hidden `<|turn>system` turn as one `<|tool>...declaration...<tool|>` block per tool. This is exactly the JSON schema `@tool` builds from `get_temperature`'s type hints and docstring, just in Gemma 4's own compact (non-JSON) syntax instead of `{...}` JSON.
- The model's turn contains three things back to back: the `<|channel>thought...<channel|>` block (its reasoning, exactly like in the prompting notebook), then `<|tool_call>call:...{...}<tool_call|>` (its decision to call `get_temperature` with `city: "Tokyo"`), then, after the tool result is inserted via `<|tool_response>...<tool_response|>`, its final natural-language answer - all inside the *same* model turn, closed by a single `<turn|>`.
- LangChain's `AIMessage.tool_calls` and the `ToolMessage` we append are really just a structured, Python-friendly view onto these `<|tool_call>` / `<|tool_response>` tokens. Whichever library you use (`ollama`, `openai`, or `langchain`), they all boil down to the same underlying raw text.

## 2. Connecting an Agent to Our MCP Server

In the previous notebook, we built a small MCP server (`mcp_server.py`, in this same folder) exposing a university library catalog. `langchain-mcp-adapters` lets a LangChain agent use those exact tools directly - no need to redefine anything.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient({
    "library": {
        "transport": "stdio",
        "command": "python",
        "args": ["mcp_server.py"],  # the exact same server from the previous notebook
    }
})

library_tools = await mcp_client.get_tools()
print([t.name for t in library_tools])

One thing to know: `MultiServerMCPClient` is **stateless by default** - each tool call opens a fresh connection (and, for our `stdio` server, launches a fresh `mcp_server.py` subprocess) rather than keeping one connection open across calls. That's fine for read-only lookups like the ones below, but it matters for `checkout_book`, which *changes* the catalog - we'll come back to that in the exercise.

In [ ]:
library_agent = create_agent(model=llm, tools=library_tools)

result = library_agent.invoke(
    {"messages": [{"role": "user", "content": "Is Deep Learning by Ian Goodfellow available in the library?"}]}
)
print(result["messages"][-1].content)

## 3. ReAct: Reasoning + Acting

**ReAct** (Yao et al., 2022, ["ReAct: Synergizing Reasoning and Acting in Language Models"](https://arxiv.org/pdf/2210.03629)) was one of the first and most influential patterns for combining reasoning with tool use: instead of jumping straight to an answer, the model interleaves **Thought** steps (reasoning about what to do next) with **Action** steps (using a tool) and **Observation** steps (the tool's result) - repeating until it has enough information to answer.

Here's the thing: every agent we've built in this course already follows this pattern. `create_agent`'s loop *is* reason-act-observe - it's just that the "reasoning" happens via native tool calling (and, for `LLM_REASONING`, actual `thinking` output) instead of the model writing out literal `"Thought: ..."` text.

Before models supported native tool calling, ReAct had to be implemented exactly that way: with a carefully written prompt asking the model to produce `Thought:` / `Action:` / `Action Input:` text, and code that *parses* that text to figure out which tool to run. LangChain still ships this classic version, in `langchain_classic`, mostly for historical/educational purposes and for models without native tool support. Let's build one, to see both the pattern and why it was eventually replaced.

In [ ]:
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate

react_prompt = PromptTemplate.from_template("""Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}""")

classic_agent = create_react_agent(llm, [get_temperature], react_prompt)
executor = AgentExecutor(agent=classic_agent, tools=[get_temperature], verbose=True, handle_parsing_errors=True)

result = executor.invoke({"input": "What is the temperature in Tokyo?"})
print()
print("Final answer:", result["output"])

With `verbose=True`, you can watch the raw back-and-forth. If you run this yourself, don't be surprised if you see something like `Invalid Format: Missing 'Action:' after 'Thought:'` in the middle — `LLM_REASONING` is trained for native tool calling, so left to its own devices it often wants to just write `get_temperature(city='Tokyo')` directly instead of following the exact `Action:` / `Action Input:` text format this classic approach depends on. `handle_parsing_errors=True` catches that, tells the model what went wrong, and lets it retry - which is exactly why this approach was fragile in practice, and why modern agents use native tool calling instead (as in Sections 1 and 2 above) rather than asking the model to format its own "function calls" as plain text.

## 4. Structured Output from an Agent

Recall `response_format` from the structured outputs notebook. Agents support it too, via `response_format=<PydanticModel>` - the final answer comes back as a validated object at `result["structured_response"]`, alongside the normal message transcript.

One thing to carry over from that notebook: a schema constraint combined with `LLM_REASONING`'s default thinking can hang (we ran into exactly this with the regex-format bug earlier). We disable thinking on the model instance for this call to avoid it.

In [ ]:
from pydantic import BaseModel


class WeatherReport(BaseModel):
    city: str
    temperature_celsius: float
    advice: str


llm_no_thinking = ChatOllama(model=LLM_REASONING, base_url=LLM_URL, temperature=0, reasoning=False)
structured_agent = create_agent(model=llm_no_thinking, tools=[get_temperature], response_format=WeatherReport)

result = structured_agent.invoke(
    {"messages": [{"role": "user", "content": "What is the temperature in Tokyo? Give me clothing advice."}]}
)
print(result["structured_response"])

## 5. Human in the Loop

Some actions shouldn't happen just because the model decided to call a tool - sending an email, charging a card, checking out a library book. `HumanInTheLoopMiddleware` pauses the agent right before a chosen tool runs, and waits for a person to approve, reject, or edit it.

This requires a `checkpointer` (so the agent's state can be paused and resumed later) - the same `InMemorySaver` + `thread_id` pattern LangChain uses for multi-turn conversations.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


@tool
def send_reminder_email(to: str, subject: str) -> str:
    """Send a reminder email to a library member."""
    return f"Email sent to {to} with subject: {subject}"


hitl_agent = create_agent(
    model=llm,
    tools=[send_reminder_email],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_reminder_email": True})],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "hitl-demo"}}
result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send a reminder email to anna@university.edu about her overdue book."}]},
    config=config,
)

print("Interrupted:", "__interrupt__" in result)
if "__interrupt__" in result:
    print(result["__interrupt__"][0].value)

The agent stopped right before calling `send_reminder_email` and handed us back a description of the pending action instead of running it. We resume the same `thread_id` with a decision - `"approve"` to let it proceed, or `"reject"` (optionally with a message explaining why) to stop it. (`"edit"` and `"respond"` are also available, for changing the tool's arguments or answering on the human's behalf without running the tool at all.)

In [ ]:
approved = hitl_agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)
print("Approved ->", approved["messages"][-1].content)

In [ ]:
config2 = {"configurable": {"thread_id": "hitl-demo-2"}}
hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send a reminder email to anna@university.edu about her overdue book."}]},
    config=config2,
)
rejected = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "Not now, wait until next week."}]}),
    config=config2,
)
print("Rejected ->", rejected["messages"][-1].content)

## Exercise: Approve a Real Checkout

`mcp_server.py` now has a fourth tool: `checkout_book(isbn, member_name)` - unlike the read-only tools from Section 2, this one actually *changes* the catalog, so it's exactly the kind of action that deserves a human in the loop before it runs.

Since this tool mutates state, and `MultiServerMCPClient` is stateless by default (a fresh `mcp_server.py` subprocess per call, as mentioned in Section 2), we need a **stateful session** so the checkout actually sticks around long enough to verify it: `async with mcp_client.session("library") as session:` keeps one connection (and one subprocess, with one in-memory copy of the catalog) alive for everything inside the block.

Steps:
1. Open a stateful session and load its tools with `load_mcp_tools(session)`.
2. Build an agent with `HumanInTheLoopMiddleware(interrupt_on={"checkout_book": True})` and a checkpointer.
3. Ask it to check out *"Campbell Biology"* for *Clara Schmidt*, approve the interrupt, then ask (in the same session) whether that book is still available - it shouldn't be.

In [ ]:
# Your code here...

<details>
<summary><b>Show solution</b></summary>

```python
from langchain_mcp_adapters.tools import load_mcp_tools

async with mcp_client.session("library") as session:
    session_tools = await load_mcp_tools(session)

    checkout_agent = create_agent(
        model=llm,
        tools=session_tools,
        middleware=[HumanInTheLoopMiddleware(interrupt_on={"checkout_book": True})],
        checkpointer=InMemorySaver(),
    )

    config = {"configurable": {"thread_id": "checkout-demo"}}
    result = await checkout_agent.ainvoke(
        {"messages": [{"role": "user", "content": "Please check out 'Campbell Biology' for Clara Schmidt."}]},
        config=config,
    )
    print("Interrupted:", "__interrupt__" in result)

    approved = await checkout_agent.ainvoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)
    print("After approval:", approved["messages"][-1].content)

    # Same session -> same subprocess -> the mutation is still there
    check = await checkout_agent.ainvoke(
        {"messages": [{"role": "user", "content": "Is Campbell Biology available right now?"}]},
        config=config,
    )
    print("Availability check:", check["messages"][-1].content)
```

</details>